# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata overview
meta = dataset.metadata  # Keep as an object, do not subscript
print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Published: {meta.date_published}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")
print(f"Cite as: {meta.cite_as}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and structure.


In [ ]:
# List all record sets, their IDs, and their fields
print("Record Sets and Fields:\n")
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', 'No description')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}")
        print(f"      Name: {field.name}")
        print(f"      Data Type: {field.data_type}")
    print('-'*60)

# Collect all record set ids for later reference
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All reference to record sets and fields uses their `@id`.


In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nFirst 3 rows for record set {record_set_id}:")
    display(df.head(3))
    print(f"Fields (columns): {df.columns.tolist()}")

# For demonstration, let's pick the first record set for EDA
main_record_set = record_set_ids[0]
main_df = dataframes[main_record_set]


## 4. Exploratory Data Analysis (EDA)
Filtering, normalization, and grouping using field `@id`s.


In [ ]:
# List fields and their data types for main_record_set:
print("Available fields in main_record_set:")
for f in dataset.metadata.record_sets_by_id[main_record_set].fields:
    print(f"Field @id: {f.id} | Name: {f.name} | DataType: {f.data_type}")

# Choose a numeric field from this list (adjust as necessary if field names differ)
# For demonstration, suppose there is a field for peptide count, e.g. 'peptide_count' or 'cr:peptide_count'
# If you see a different field in your schema, substitute its @id below.
numeric_field_id = None
for f in dataset.metadata.record_sets_by_id[main_record_set].fields:
    if 'count' in f.id or f.data_type in ('Integer', 'Float', 'Number'):
        numeric_field_id = f.id
        break
if numeric_field_id is None:
    numeric_field_id = main_df.select_dtypes(include=['int', 'float']).columns[0]

print(f"\nUsing numeric field for EDA: {numeric_field_id}")

# Filter where numeric field > threshold
threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (showing up to 5):")
display(filtered_df.head())

# Normalize the numeric field
normalized_col = f"{numeric_field_id}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print("Normalized column (showing 5):")
display(filtered_df[[numeric_field_id, normalized_col]].head())

# Try grouping by a categorical field (e.g. accession or description)
group_field_id = None
categorical_candidates = [f.id for f in dataset.metadata.record_sets_by_id[main_record_set].fields if f.data_type == 'Text' or 'accession' in f.id or 'description' in f.id]
for c in categorical_candidates:
    if c in filtered_df.columns and filtered_df[c].nunique() < 20 and filtered_df[c].nunique() > 1:
        group_field_id = c
        break
# If no small cardinality, just use the first text field
if group_field_id is None and len(categorical_candidates) > 0:
    group_field_id = categorical_candidates[0]

if group_field_id is not None:
    print(f"\nGrouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
    display(grouped_df.head(10))
else:
    print("No suitable group field (")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot for normalized values, if available
if normalized_col in filtered_df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(y=filtered_df[normalized_col])
    plt.title(f"Boxplot of normalized {numeric_field_id} (filtered > {threshold})")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load a Croissant-packaged dataset using `mlcroissant`, inspect its schema, extract record sets by `@id`, filter and normalize numeric fields, and perform simple visualizations. Always reference entities in your analyses using their `@id` for reproducibility and to match schema semantics. Continue exploring the dataset, adjusting filters, and analyzing relationships between fields as needed.